# Czech 2021 parliamentary elections
This example shows a complicated multi-constituency proportional system that was newly introduced the same year.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import csv
import decimal

sys.path.append(os.path.join('..', '..'))
import votelib.candidate
import votelib.evaluate.core
import votelib.evaluate.threshold
import votelib.evaluate.proportional

## Evaluator construction
First, we construct the evaluator.

The country is divided into fourteen regions that form separate constituencies with separate regional lists. The number of seats distributed within each region is determined from the national total of 200 seats by the Hare quota computed on the national vote count total. There is also a national 5% electoral threshold.

In [3]:
eliminator = votelib.evaluate.threshold.RelativeThreshold(
    decimal.Decimal('.05'), accept_equal=True
)
apportioner = votelib.evaluate.proportional.LargestRemainder('hare')

The evaluator has two stages. In the first stage, seats are allocated within regions using the Imperiali quota. The awarded seat counts are rounded down - there is no largest remainder closeup, and so not all seats are awarded; these are transferred to the second stage.

In [4]:
regional_evaluator = votelib.evaluate.proportional.QuotaDistributor('imperiali', on_overaward='subtract')

stage1_evaluator = votelib.evaluate.core.ByConstituency(regional_evaluator)

In the second stage, the votes "unused" in the regional stage are summed across regions and the remaining seats allocated according to them using the Droop quota, now using the largest remainder rule to distribute all seats. The second stage seats are redistributed back to the regions according to the regional remainders and the candidates elected are determined through open lists, but we will not go into that detail here.

In [5]:
stage2_evaluator = votelib.evaluate.core.RemovedApportionment(
    votelib.evaluate.core.ByParty(
        votelib.evaluate.proportional.LargestRemainder('droop')
    )
)

evaluator = votelib.evaluate.core.FixedSeatCount(
    votelib.evaluate.core.PreApportioned(
        evaluator=votelib.evaluate.core.Conditioned(
            evaluator=votelib.evaluate.core.UnusedVotesDistributor(
                [stage1_evaluator, stage2_evaluator],
                quota_functions=[regional_evaluator.quota_function],
                depth=2
            ),
            eliminator=eliminator,
            depth=2,
        ),
        apportioner=apportioner,
    ),
    200
)

country_evaluator = votelib.evaluate.core.PostConverted(evaluator, votelib.convert.MergedDistributions())

## Vote loading
Now we load the vote counts, per region.

In [11]:
fpath = os.path.join('..', '..', 'krajvolby_2024.csv')
with open(fpath, encoding='cp1250') as infile:
    rows = list(csv.reader(infile, delimiter=';'))
region_names = rows[0][1:]
votes = {region: {} for region in region_names}
for row in rows[1:]:
    party = row[0]
    for regname, n_votes in zip(region_names, row[1:]):
        votes[regname][party] = int(n_votes)
print(dict(sorted(votes['Královéhradecký kraj'].items(), key=lambda x: x[1], reverse=True)))

{'ANO': 50573, 'Spolu': 39036, 'STAN': 17790, 'SOCDEM': 10323, 'SPD': 9400, 'Stačilo': 8390, 'Piráti': 5798, 'Zelení': 2723, 'Ostatní': 1567, 'NK': 820, 'Přísaha': 0}


## Performing the evaluation
When the evaluator is set up correctly, obtaining the result is simple.

In [12]:
evaluator.evaluate(votes)

{'Jihočeský kraj': {'ANO': 4, 'Spolu': 8, 'Stačilo': 1},
 'Karlovarský kraj': {'ANO': 3, 'STAN': 1},
 'Liberecký kraj': {'ANO': 3, 'STAN': 4, 'Spolu': 1},
 'Pardubický kraj': {'ANO': 5, 'Stačilo': 1, 'Spolu': 2, 'SPD': 1, 'STAN': 1},
 'Moravskoslezský kraj': {'ANO': 11,
  'Stačilo': 2,
  'Spolu': 3,
  'SPD': 1,
  'STAN': 3,
  'Piráti': 1},
 'Jihomoravský kraj': {'ANO': 8,
  'Stačilo': 1,
  'Spolu': 11,
  'SPD': 1,
  'STAN': 1,
  'Piráti': 1},
 'Zlínský kraj': {'ANO': 5, 'Spolu': 4, 'STAN': 1, 'SPD': 1},
 'Středočeský kraj': {'ANO': 10,
  'Stačilo': 1,
  'Spolu': 5,
  'Piráti': 1,
  'SPD': 1,
  'STAN': 8},
 'Plzeňský kraj': {'ANO': 5, 'Spolu': 3, 'STAN': 2, 'Piráti': 1, 'SPD': 1},
 'Kraj Vysočina': {'ANO': 4, 'Spolu': 4, 'STAN': 1},
 'Královéhradecký kraj': {'ANO': 5, 'Spolu': 4, 'STAN': 1, 'SPD': 1},
 'Olomoucký kraj': {'ANO': 5, 'Spolu': 2, 'SPD': 1, 'STAN': 2, 'Stačilo': 1},
 'Ústecký kraj': {'ANO': 7, 'Stačilo': 1, 'Spolu': 2, 'SPD': 1, 'STAN': 2},
 'Praha': {'ANO': 5,
  'Stačilo': 

In [13]:
print(country_evaluator.evaluate(votes))

{'ANO': 80, 'Spolu': 60, 'Stačilo': 10, 'STAN': 31, 'SPD': 11, 'Piráti': 8}


In [17]:
for pty, m in country_evaluator.evaluate(votes).items():
    print(pty, m)

ANO 80
Spolu 60
Stačilo 10
STAN 31
SPD 11
Piráti 8


In [14]:
# with zeleni
modif_votes = {
    reg: {
        **{pty: n for pty, n in regvotes.items() if pty not in ["Piráti", "Zelení"]},
        'PirZel': regvotes.get('Piráti', 0)+regvotes.get('Zelení', 0)
    }
    for reg, regvotes in votes.items()
}
modif_votes['Praha']

{'ANO': 55794,
 'Stačilo': 20329,
 'Spolu': 120993,
 'SOCDEM': 6479,
 'SPD': 21443,
 'STAN': 45656,
 'NK': 6713,
 'Přísaha': 28862,
 'Ostatní': 6521,
 'PirZel': 58417}

In [15]:
evaluator.evaluate(modif_votes)

{'Jihočeský kraj': {'ANO': 4, 'Spolu': 8, 'Stačilo': 1},
 'Karlovarský kraj': {'ANO': 3, 'STAN': 1},
 'Liberecký kraj': {'ANO': 3, 'STAN': 3, 'Spolu': 1},
 'Pardubický kraj': {'ANO': 5, 'Stačilo': 1, 'Spolu': 2, 'SPD': 1, 'STAN': 1},
 'Moravskoslezský kraj': {'ANO': 11,
  'Stačilo': 2,
  'Spolu': 3,
  'SPD': 1,
  'STAN': 3,
  'PirZel': 1},
 'Jihomoravský kraj': {'ANO': 8,
  'Stačilo': 1,
  'Spolu': 11,
  'SPD': 1,
  'STAN': 1,
  'PirZel': 1},
 'Zlínský kraj': {'ANO': 5, 'Spolu': 4, 'STAN': 1, 'SPD': 1},
 'Středočeský kraj': {'ANO': 10,
  'Stačilo': 1,
  'Spolu': 5,
  'SPD': 1,
  'STAN': 8,
  'PirZel': 1},
 'Plzeňský kraj': {'ANO': 5,
  'Spolu': 3,
  'STAN': 2,
  'Stačilo': 1,
  'SPD': 1,
  'PirZel': 1},
 'Kraj Vysočina': {'ANO': 4, 'Spolu': 4, 'STAN': 1},
 'Královéhradecký kraj': {'ANO': 5,
  'Spolu': 3,
  'STAN': 2,
  'SPD': 1,
  'PirZel': 1},
 'Olomoucký kraj': {'ANO': 5, 'Spolu': 2, 'SPD': 1, 'STAN': 2, 'Stačilo': 1},
 'Ústecký kraj': {'ANO': 6, 'Stačilo': 1, 'Spolu': 2, 'SPD': 1, '

In [16]:
print(country_evaluator.evaluate(modif_votes))

{'ANO': 79, 'Spolu': 59, 'Stačilo': 10, 'STAN': 31, 'SPD': 11, 'PirZel': 10}


In [22]:
stage1_res = votelib.evaluate.core.FixedSeatCount(
    votelib.evaluate.core.PreApportioned(
        evaluator=votelib.evaluate.core.Conditioned(
            evaluator=stage1_evaluator,
            eliminator=eliminator,
            depth=2,
        ),
        apportioner=apportioner,
    ),
    200
).evaluate(votes)
stage1_res

{'Praha': {'ANO': 5,
  'Stačilo': 1,
  'Spolu': 11,
  'Piráti': 4,
  'SPD': 2,
  'STAN': 4},
 'Středočeský kraj': {'ANO': 9,
  'Stačilo': 1,
  'Spolu': 5,
  'Piráti': 1,
  'SPD': 1,
  'STAN': 7},
 'Jihočeský kraj': {'ANO': 3, 'Spolu': 8},
 'Plzeňský kraj': {'ANO': 5, 'Spolu': 3, 'STAN': 1},
 'Karlovarský kraj': {'ANO': 3, 'STAN': 1},
 'Ústecký kraj': {'ANO': 6, 'Stačilo': 1, 'Spolu': 2, 'SPD': 1, 'STAN': 2},
 'Liberecký kraj': {'ANO': 3, 'STAN': 3},
 'Královéhradecký kraj': {'ANO': 5, 'Spolu': 3, 'STAN': 1},
 'Pardubický kraj': {'ANO': 5, 'Stačilo': 1, 'Spolu': 2, 'SPD': 1, 'STAN': 1},
 'Kraj Vysočina': {'ANO': 4, 'Spolu': 4, 'STAN': 1},
 'Jihomoravský kraj': {'ANO': 8,
  'Stačilo': 1,
  'Spolu': 11,
  'SPD': 1,
  'STAN': 1},
 'Olomoucký kraj': {'ANO': 5, 'Spolu': 2, 'SPD': 1, 'STAN': 2},
 'Zlínský kraj': {'ANO': 5, 'Spolu': 3, 'STAN': 1},
 'Moravskoslezský kraj': {'ANO': 11,
  'Stačilo': 1,
  'Spolu': 3,
  'SPD': 1,
  'STAN': 3}}

In [23]:
sum(sum(reg.values()) for reg in stage1_res.values())

181